# Step 2 — Character Segmentation

Goal: take the binary image from preprocessing and cut out each character separately.
We need bounding boxes around each symbol so we can feed them to a classifier later.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('..')

from src.preprocess import preprocess

def show(title, img, cmap='gray'):
    plt.figure(figsize=(8, 3))
    plt.imshow(img, cmap=cmap)
    plt.title(title)
    plt.axis('off')
    plt.show()

## Load preprocessed image

In [ ]:
binary = preprocess('../data/raw_samples/synthetic_eq1.png')
show('binary input', binary)

## Find contours

Find all the blobs in the binary image. Each contour = some connected white region.

In [ ]:
contours, hierarchy = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f'found {len(contours)} contours')

# draw them to see what we got
vis = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
cv2.drawContours(vis, contours, -1, (0, 255, 0), 2)
show('all contours', cv2.cvtColor(vis, cv2.COLOR_BGR2RGB))

## Get bounding boxes

Each contour gives us a bounding rectangle. But some are tiny noise specks — filter by area.

In [ ]:
boxes = []
img_area = binary.shape[0] * binary.shape[1]
min_area = img_area * 0.001  # ignore anything smaller than 0.1% of image

for cnt in contours:
    area = cv2.contourArea(cnt)
    if area < min_area:
        continue
    x, y, w, h = cv2.boundingRect(cnt)
    boxes.append((x, y, w, h))

print(f'{len(boxes)} boxes after filtering (min area: {min_area:.0f}px)')

# draw boxes
vis2 = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in boxes:
    cv2.rectangle(vis2, (x, y), (x+w, y+h), (0, 255, 0), 2)
show('bounding boxes (filtered)', cv2.cvtColor(vis2, cv2.COLOR_BGR2RGB))

## Merge overlapping boxes

Problem: `=` sign is two separate horizontal bars → two contours.
Same issue with `i`, `j`, `!` etc — multiple parts that belong to one symbol.

Fix: if two boxes overlap horizontally (their x ranges intersect), merge them into one.

In [ ]:
def merge_overlapping_boxes(boxes, overlap_thresh=0.5):
    """merge boxes that overlap horizontally — handles = sign, i dot, etc"""
    if not boxes:
        return []

    # sort by x position
    boxes = sorted(boxes, key=lambda b: b[0])
    merged = [list(boxes[0])]

    for box in boxes[1:]:
        x, y, w, h = box
        prev = merged[-1]
        px, py, pw, ph = prev

        # check horizontal overlap
        overlap_start = max(px, x)
        overlap_end = min(px + pw, x + w)
        overlap_width = max(0, overlap_end - overlap_start)
        min_width = min(pw, w)

        if min_width > 0 and overlap_width / min_width > overlap_thresh:
            # merge — expand prev box to cover both
            new_x = min(px, x)
            new_y = min(py, y)
            new_w = max(px + pw, x + w) - new_x
            new_h = max(py + ph, y + h) - new_y
            merged[-1] = [new_x, new_y, new_w, new_h]
        else:
            merged.append(list(box))

    return [tuple(b) for b in merged]

merged_boxes = merge_overlapping_boxes(boxes)
print(f'{len(merged_boxes)} symbols after merging')

In [ ]:
# draw merged boxes
vis3 = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
for (x, y, w, h) in merged_boxes:
    cv2.rectangle(vis3, (x, y), (x+w, y+h), (0, 255, 0), 2)
show('merged boxes', cv2.cvtColor(vis3, cv2.COLOR_BGR2RGB))

## Sort left to right

We need the characters in reading order. Sort by x coordinate.

In [ ]:
sorted_boxes = sorted(merged_boxes, key=lambda b: b[0])

# draw with numbering
vis4 = cv2.cvtColor(binary, cv2.COLOR_GRAY2BGR)
for i, (x, y, w, h) in enumerate(sorted_boxes):
    cv2.rectangle(vis4, (x, y), (x+w, y+h), (0, 255, 0), 2)
    cv2.putText(vis4, str(i), (x, y - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
show('sorted L-R', cv2.cvtColor(vis4, cv2.COLOR_BGR2RGB))

## Extract individual characters

Crop each box, pad to square, resize to 45x45. The padding keeps aspect ratio so characters dont get stretched.

In [ ]:
def extract_characters(binary_img, boxes, target_size=45, padding=4):
    """crop each box from binary image, pad to square, resize"""
    chars = []
    for (x, y, w, h) in boxes:
        # crop with a small margin
        y1 = max(0, y - padding)
        y2 = min(binary_img.shape[0], y + h + padding)
        x1 = max(0, x - padding)
        x2 = min(binary_img.shape[1], x + w + padding)
        crop = binary_img[y1:y2, x1:x2]

        # pad to square
        ch, cw = crop.shape[:2]
        side = max(ch, cw)
        square = np.zeros((side, side), dtype=np.uint8)
        off_y = (side - ch) // 2
        off_x = (side - cw) // 2
        square[off_y:off_y+ch, off_x:off_x+cw] = crop

        # resize
        resized = cv2.resize(square, (target_size, target_size), interpolation=cv2.INTER_AREA)
        chars.append(resized)

    return chars

characters = extract_characters(binary, sorted_boxes)
print(f'extracted {len(characters)} characters')

In [ ]:
# show all extracted characters side by side
fig, axes = plt.subplots(1, len(characters), figsize=(2 * len(characters), 2))
if len(characters) == 1:
    axes = [axes]
for i, char_img in enumerate(characters):
    axes[i].imshow(char_img, cmap='gray')
    axes[i].set_title(f'#{i}')
    axes[i].axis('off')
plt.suptitle('extracted characters')
plt.tight_layout()
plt.show()

## Save segments

Save each character crop to disk so we can inspect them.

In [ ]:
import os
out_dir = '../data/segments'
os.makedirs(out_dir, exist_ok=True)

for i, char_img in enumerate(characters):
    path = os.path.join(out_dir, f'char_{i}.png')
    cv2.imwrite(path, char_img)
    print(f'saved {path}')

print(f'\ndone — {len(characters)} segments saved')

## Full segmentation function

Wrap everything into one call — this is what goes into `src/segment.py`.

In [ ]:
def segment(binary_img, min_area_ratio=0.001, overlap_thresh=0.5, target_size=45, padding=4):
    """takes binary image, returns list of character images sorted left to right"""
    # find contours
    contours, _ = cv2.findContours(binary_img, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # bounding boxes, filter small ones
    img_area = binary_img.shape[0] * binary_img.shape[1]
    min_area = img_area * min_area_ratio
    boxes = []
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        boxes.append(cv2.boundingRect(cnt))

    # merge overlapping (for = sign etc)
    merged = merge_overlapping_boxes(boxes, overlap_thresh)

    # sort left to right
    merged = sorted(merged, key=lambda b: b[0])

    # extract and resize
    chars = extract_characters(binary_img, merged, target_size, padding)

    return chars, merged

# test it
chars, boxes = segment(binary)
print(f'segment() returned {len(chars)} characters')